# генерация

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def gen(N, x_min=0, x_max=10, y_min=0, y_max=10, straight_ratio=0.5, rand_seed=40, decimals=None):

    """
    Генерирует данные в формате [x1, y1, x2, y2, x3, y3] для N точек.
    x_min, x_max, y_min, y_max - пределы значений
    straight_ratio определяет процент прямых линий
    rand_seed задаёт сид для локального генератора случайных чисел
    decimals задаёт округление значений до указанного знака

    """

    rng = np.random.default_rng(rand_seed)

    N_straight = int(N * straight_ratio)
    N_avg = N - N_straight

    # прямые линии
    x_strait = rng.uniform(x_min, x_max, N_straight)
    y_strait = rng.uniform(y_min, y_max, N_straight)
    strait_data = np.column_stack([x_strait, y_strait, x_strait, y_strait, x_strait, y_strait])

    # линии под углом
    x1 = rng.uniform(x_min, x_max, N_avg)
    y1 = rng.uniform(y_min, y_max, N_avg)
    x3 = rng.uniform(x_min, x_max, N_avg)
    y3 = rng.uniform(y_min, y_max, N_avg)

    x2 = (x1 + x3) / 2
    y2 = (y1 + y3) / 2
    avg_data = np.column_stack([x1, y1, x2, y2, x3, y3])

    # Объединение
    if N_straight == 0:
        data = avg_data
    elif N_avg == 0:
        data = strait_data
    else:
        data = np.vstack([strait_data, avg_data])

    # Округление
    if decimals is not None:
        data = np.round(data, decimals=decimals)

    return data


def gen_shift(data, shift=0.01, loc=3):
    """создаёт сдвиг указанного размера на указанной позиции"""
    data = data.copy()
    data[:, loc] += shift
    return data


def plot_dist(data, bins=30, figsize=(12, 4)):
    y1, y2, y3 = data[:, 1], data[:, 3], data[:, 5]

    fig, axs = plt.subplots(1, 3, figsize=figsize, sharey=True)
    colors = ['skyblue', 'lightgreen', 'salmon']
    labels = ['y1', 'y2', 'y3']

    for ax, y, color, label in zip(axs, [y1, y2, y3], colors, labels):
        ax.hist(y, bins=bins, color=color, edgecolor='black', alpha=0.8)
        ax.set_xlabel(label)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_title(f'Распределение {label}')

    plt.tight_layout()
    plt.show()

# WGAN

In [ ]:
pip install comet_ml

In [ ]:
import os
import json
from comet_ml import start
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt




In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.shift = nn.Parameter(torch.zeros(6), requires_grad=True)

    def forward(self, x_shifted):
        return x_shifted + self.shift

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [ ]:
def compute_gradient_penalty(discriminator, real_samples, fake_samples, device, lambda_gp=10.0):
    """
    Вычисляет gradient penalty для WGAN-GP.
    Возвращает: (значение штрафа, средняя норма градиента)
    """
    batch_size = real_samples.size(0)

    #Интерполяция (ε ~ U[0, 1])
    epsilon = torch.rand(batch_size, 1, device=device)
    epsilon = epsilon.expand_as(real_samples)

    #Интерполированные сэмплы
    interpolates = (epsilon * real_samples + (1 - epsilon) * fake_samples).requires_grad_(True)

    d_interpolates = discriminator(interpolates)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates, device=device),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(batch_size, -1)
    grad_norm = gradients.norm(2, dim=1)

    gradient_penalty = lambda_gp * ((grad_norm - 1) ** 2).mean()

    return gradient_penalty, grad_norm

In [ ]:
def get_config(**kwargs):
    config = {
        "mode": "independent",
        "N": 500000,
        "straight_ratio": 1.0,
        "decimals": 4,

        "shift_value": 0.01,
        "shift_loc": 3,

        "epochs": 800,
        "batch_size": 4096,
        "n_critic": 10,

        "lr_G": 5e-6,
        "lr_D": 1e-4,
        "lambda_gp": 10.0,

        "beta1": 0.5,
        "beta2": 0.9,

        "ema_alpha": 0.995,

        "seed_real": 1,
        "seed_fake": 2,
    }

    config.update(kwargs)
    return config

In [ ]:
def format_float(x):
    if x == 0:
        return "0"

    if abs(x) < 1e-3 or abs(x) >= 1e3:
        s = f"{x:.0e}"
        s = s.replace("-", "m")
        return s

    s = str(x)
    s = s.replace(".", "p")
    return s

In [ ]:
def build_name(cfg):
    return (
        f"WGAN_GP_{cfg['mode']}"
        f"_N{cfg['N']}"
        f"_sr{format_float(cfg['straight_ratio'])}"
        f"_shift{format_float(cfg['shift_value'])}"
        f"_dec{cfg['decimals']}"
        f"_epochs{cfg['epochs']}"
        f"_bs{cfg['batch_size']}"
        f"_nc{cfg['n_critic']}"
        f"_lrG{format_float(cfg['lr_G'])}"
        f"_lrD{format_float(cfg['lr_D'])}"
        f"_gp{format_float(cfg['lambda_gp'])}"
    )

In [ ]:
def generate_data(cfg):
    real = gen(
        N=cfg["N"],
        straight_ratio=cfg["straight_ratio"],
        rand_seed=cfg["seed_real"],
        decimals=cfg["decimals"]
    )

    if cfg["mode"] == "shared":
        # fake = real + shift
        fake = gen_shift(real, cfg["shift_value"], cfg["shift_loc"])

    elif cfg["mode"] == "independent":
        # fake генерируется отдельно + shift
        fake_base = gen(
            N=cfg["N"],
            straight_ratio=cfg["straight_ratio"],
            rand_seed=cfg["seed_fake"],
            decimals=cfg["decimals"]
        )
        fake = gen_shift(fake_base, cfg["shift_value"], cfg["shift_loc"])

    else:
        raise ValueError("mode must be 'shared' or 'independent'")

    return real, fake

In [ ]:
def create_loaders(real, fake, cfg):
    real_tensor = torch.tensor(real, dtype=torch.float32)
    fake_tensor = torch.tensor(fake, dtype=torch.float32)

    real_loader = DataLoader(TensorDataset(real_tensor), batch_size=cfg["batch_size"], shuffle=True)
    fake_loader = DataLoader(TensorDataset(fake_tensor), batch_size=cfg["batch_size"], shuffle=True)

    return real_loader, fake_loader

In [ ]:
def train(cfg, experiment=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    real, fake = generate_data(cfg)
    real_loader, fake_loader = create_loaders(real, fake, cfg)

    G = Generator().to(device)
    D = Discriminator().to(device)

    opt_G = optim.Adam(G.parameters(), lr=cfg["lr_G"], betas=(cfg["beta1"], cfg["beta2"]))
    opt_D = optim.Adam(D.parameters(), lr=cfg["lr_D"], betas=(cfg["beta1"], cfg["beta2"]))

    losses_G, losses_D = [], []
    shift_history = []
    ema_history = []

    ema = np.zeros(6)

    true_values = np.zeros(6)
    true_values[cfg["shift_loc"]] = -cfg["shift_value"]

    for epoch in range(cfg["epochs"]):

        for (r,), (f,) in zip(real_loader, fake_loader):
            r, f = r.to(device), f.to(device)
            step = len(losses_G)

            # D
            for _ in range(cfg["n_critic"]):
                opt_D.zero_grad()

                real_pred = D(r)
                fake_data = G(f).detach()
                fake_pred = D(fake_data)

                loss_D = -real_pred.mean() + fake_pred.mean()

                gp, grad_norm = compute_gradient_penalty(
                    D, r, fake_data, device, cfg["lambda_gp"]
                )
                if experiment:
                    experiment.log_metric("gp", gp.item(), step=step)
                    experiment.log_metric("grad_norm", grad_norm.mean().item(), step=step)

                total_D = loss_D + gp
                total_D.backward()
                opt_D.step()

            # G
            opt_G.zero_grad()
            gen_data = G(f)
            loss_G = -D(gen_data).mean()
            loss_G.backward()
            opt_G.step()

            losses_D.append(total_D.item())
            losses_G.append(loss_G.item())
            if experiment:
                experiment.log_metric("loss_D", total_D.item(), step=step)
                experiment.log_metric("loss_G", loss_G.item(), step=step)

        shift = G.shift.detach().cpu().numpy()
        shift_history.append(shift.copy())

        ema = (cfg["ema_alpha"] * ema + (1 - cfg["ema_alpha"]) * shift)
        ema_history.append(ema.copy())

        if experiment:
            epoch_num = len(shift_history)


            for i in range(6):
                experiment.log_metric(f"shift_{i}", shift[i], step=epoch_num)

            experiment.log_metric("shift_y2", shift[3], step=epoch_num)
            for i in range(6):
                experiment.log_metric(f"ema_shift_{i}", ema[i], step=epoch_num)

            error = shift[3] - true_values[3]
            experiment.log_metric("error_y2", error, step=epoch_num)

        print(f"Epoch {epoch+1}/{cfg['epochs']} | shift_y2={shift[3]:.6f}")

    return {
        "G": G,
        "losses_G": np.array(losses_G),
        "losses_D": np.array(losses_D),
        "shift_history": np.array(shift_history),
        "ema_history": np.array(ema_history),
    }


In [ ]:
def save_results(results, cfg, name):
    os.makedirs("experiments", exist_ok=True)
    path = os.path.join("experiments", name)
    os.makedirs(path, exist_ok=True)

    np.save(os.path.join(path, "shift_history.npy"), results["shift_history"])
    np.save(os.path.join(path, "ema_history.npy"), results["ema_history"])
    np.save(os.path.join(path, "loss_G.npy"), results["losses_G"])
    np.save(os.path.join(path, "loss_D.npy"), results["losses_D"])

    with open(os.path.join(path, "config.json"), "w") as f:
        json.dump(cfg, f, indent=4)

    print("Saved to:", path)

In [ ]:
def plot_results(results, log_to_comet=False, experiment=None):
    shift_history = results["shift_history"]
    ema_history = results["ema_history"]
    losses_G = results["losses_G"]
    losses_D = results["losses_D"]
    cfg = results["config"]

    labels = ['x1', 'y1', 'x2', 'y2', 'x3', 'y3']

    true_values = np.zeros(6)
    true_values[cfg["shift_loc"]] = -cfg["shift_value"]

    ymin = min(shift_history.min(), true_values.min()) - 0.001
    ymax = max(shift_history.max(), true_values.max()) + 0.001


    # SHIFT
    fig, axs = plt.subplots(2, 3, figsize=(14, 8))
    axs = axs.flatten()

    for i, ax in enumerate(axs):
        ax.plot(shift_history[:, i], label="learned")
        ax.axhline(true_values[i], color='red', linestyle='--', label="true")

        ax.set_ylim(ymin, ymax)
        ax.set_title(labels[i])
        ax.grid(True)
        ax.legend()

    plt.suptitle("Shift learning")
    plt.tight_layout()

    if log_to_comet and experiment:
        experiment.log_figure("shift_all_params", fig)

    plt.show()


    # EMA
    fig2, axs2 = plt.subplots(2, 3, figsize=(14, 8))
    axs2 = axs2.flatten()

    for i, ax in enumerate(axs2):
        ax.plot(shift_history[:, i], alpha=0.4, label="raw")
        ax.plot(ema_history[:, i], linewidth=2, label="ema")
        ax.axhline(true_values[i],linestyle='--', color='red', label="true")

        ax.set_ylim(ymin, ymax)
        ax.set_title(labels[i])
        ax.grid(True)
        ax.legend()

    plt.suptitle("EMA convergence")
    plt.tight_layout()

    if log_to_comet and experiment:
        experiment.log_figure("ema_all_params", fig2)

    plt.show()


    # LOSSES
    fig3 = plt.figure(figsize=(10, 4))
    plt.plot(losses_D, label="D")
    plt.plot(losses_G, label="G")

    plt.legend()
    plt.grid()
    plt.title("Losses")

    if log_to_comet and experiment:
        experiment.log_figure("losses", fig3)

    plt.show()

In [ ]:
def run_experiment(use_comet=True, plot=True, **kwargs):
    cfg = get_config(**kwargs)
    name = build_name(cfg)

    if use_comet:
        experiment = start(
            api_key="",         #your api
            project_name="",    #your project name
            workspace=""        #your workspace
        )
        experiment.set_name(name)
        experiment.log_parameters(cfg)
    else:
        experiment = None

    print("Running:", name)

    results = train(cfg, experiment)
    results["config"] = cfg

    save_results(results, cfg, name)

    if plot:
        plot_results(results, log_to_comet=use_comet, experiment=experiment)

    if use_comet:
        experiment.end()

    return results

In [ ]:
res = run_experiment(
    mode="independent",
    N=500000,
    straight_ratio=0.5,
    shift_value=0.01,
    epochs=800,
    batch_size=4096,
    n_critic=10,
    lr_G=5e-6,
    lr_D=1e-4,
    lambda_gp=10.0
)